# N2H validation in yeast

In [1]:
import glob
import xlrd
import pandas as pd
import numpy as np
import os

In [2]:
inputs_dir = "../1_inputs/N2H_validation"
allele_inputs = "../../../1_allele_collection/1_inputs"
outputs_dir = "../3_outputs/N2H_validation/yeast_N2H"

nlr_detection_threshold = 2.23

## Wild type pairwise test

In [3]:
# Load in raw data for wild type pairwise test (apwt) from the yeast N2H validation assay
res = {}
for elem in glob.glob(f'{inputs_dir}/yeast_N2H/20210302_yN2H_WT_PWT/data/*'):
    if 'Plate' in elem:
        retest_pla = elem.split("data/", 1)[-1]
        retest_pla = int(retest_pla.split('_')[1][-2:])

        
        df = pd.read_excel(elem, engine='xlrd', skiprows=10, skipfooter=35)
        df.rename(columns={c:i+1 for i,c in enumerate(df.columns)}, inplace=True)
        
        value_d = dict()
        for l, (i, row) in zip(['A','B','C','D','E','F','G','H'], df.iterrows()):
            for i, value in enumerate(row):
                key = (retest_pla,f'{l}{str(i+1).zfill(2)}')
                
                res[key] = value

*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: wi

In [4]:
# Read in the plate layout file
yN2H_retest = pd.read_csv(f"{inputs_dir}/yeast_N2H/mapping_yn2h.csv")
col = ['ad_orf_id', 'ad_symbol', 'db_orf_id', 'db_symbol', 'retest_pla', 'retest_pos', 'standard_batch', 'variant_id']
yN2H_retest = yN2H_retest[col]

yN2H_retest = yN2H_retest[yN2H_retest['standard_batch'] == "yN2Hwtpwt"]
yN2H_retest['yN2H_score'] = [res[(pla, pos)] for pla, pos in zip(yN2H_retest.retest_pla, yN2H_retest.retest_pos)]

In [5]:
# Calculate the normalized luminescnence ratio
empty_N1_d = dict(zip(yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].db_orf_id, yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].yN2H_score))
empty_N2_d = dict(zip(yN2H_retest[yN2H_retest.variant_id == 'empty_N2'].ad_orf_id, yN2H_retest[yN2H_retest.variant_id == 'empty_N2'].yN2H_score))

yN2H_normalized = yN2H_retest[yN2H_retest.variant_id.isin(['wt','RRS','PRS','litbm'])].copy()
yN2H_normalized['pair'] = [tuple(sorted([x,y])) for x, y in zip(yN2H_normalized.ad_orf_id, yN2H_normalized.db_orf_id)]
yN2H_normalized['variant_idv2'] = yN2H_normalized.variant_id.apply(lambda x: x if x !=  'litbm' else 'wt')
yN2H_normalized['empty_N1'] = yN2H_normalized.db_orf_id.map(empty_N1_d)
yN2H_normalized['empty_N2'] = yN2H_normalized.ad_orf_id.map(empty_N2_d)
yN2H_normalized['NLR'] = [N1N2/max([N1F2,N2F1]) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.yN2H_score, yN2H_normalized.empty_N1, yN2H_normalized.empty_N2)]
yN2H_normalized['detected'] = yN2H_normalized.NLR.apply(lambda x: x > yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.max())
yN2H_normalized['detectedv2'] = yN2H_normalized.NLR.apply(lambda x: x > sorted(yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.tolist())[-4])
yN2H_normalized['NLR_log2'] = [np.log2(N1N2/max([N1F2,N2F1])) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.yN2H_score, yN2H_normalized.empty_N1, yN2H_normalized.empty_N2)]

In [6]:
# Count the number detected by gene for different thresholds
yN2H_normalized['detected_0'] = yN2H_normalized.NLR.apply(lambda x: x > sorted(yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.tolist())[-1])
yN2H_normalized['detected_1'] = yN2H_normalized.NLR.apply(lambda x: x > sorted(yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.tolist())[-2])
yN2H_normalized['detected_2'] = yN2H_normalized.NLR.apply(lambda x: x > sorted(yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.tolist())[-3])
yN2H_normalized['detected_3'] = yN2H_normalized.NLR.apply(lambda x: x > sorted(yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.tolist())[-4])


by_gene_df = pd.DataFrame([pd.Series(dict(yN2H_normalized[(yN2H_normalized.variant_idv2 == 'wt')].db_symbol.value_counts())),
                           pd.Series(yN2H_normalized[yN2H_normalized.detected_0 & (yN2H_normalized.variant_idv2 == 'wt')].db_symbol.value_counts()),
                           pd.Series(yN2H_normalized[yN2H_normalized.detected_1 & (yN2H_normalized.variant_idv2 == 'wt')].db_symbol.value_counts()),
                           pd.Series(yN2H_normalized[yN2H_normalized.detected_2 & (yN2H_normalized.variant_idv2 == 'wt')].db_symbol.value_counts()),
                           pd.Series(yN2H_normalized[yN2H_normalized.detected_3 & (yN2H_normalized.variant_idv2 == 'wt')].db_symbol.value_counts())
                           
                          ]).T


by_gene_df.columns = ['y2h', 'yn2h_0','yn2h_1','yn2h_2', 'yn2h_3']
by_gene_df.fillna(0, inplace=True)
by_gene_df.sort_values(by='yn2h_0', ascending=False, inplace=True)

In [7]:
# Write out files
yN2H_normalized.to_csv(f'{outputs_dir}/yN2H_wtpwt_raw.csv')
by_gene_df.to_csv(f'{outputs_dir}/yN2H_wtpwt_counts.csv')

col = ['ad_orf_id', 'ad_symbol', 'db_orf_id', 'db_symbol', 'yN2H_score', 'variant_id',
       'variant_idv2', 'empty_N1', 'empty_N2', 'NLR',
       'NLR_log2', 'detected_3']
yN2H_normalized[(yN2H_normalized.detected_3 == True)&(yN2H_normalized.variant_idv2 == 'wt')][col].to_csv(f'{outputs_dir}/yN2H_wtpwt_positive_PPIs.csv') 

## Allele pairwise test

In [8]:
# Load in raw data for allele pairwise test (apwt) from the yeast N2H validation assay
res = {}
for f in glob.glob(f'{inputs_dir}/yeast_N2H/20210331_yN2H_APWT/data/*.xls'):
    

    retest_pla = int(f.split('_')[-4].split('Plate')[-1])

    if  retest_pla < 38:
        replicate = 1
    else:
        retest_pla -= 37
        replicate = 2

    workbook = xlrd.open_workbook(f, encoding_override='iso-8859-1') 
    df = pd.read_excel(workbook, skiprows=10, skipfooter=35)
    df.rename(columns={c:i+1 for i,c in enumerate(df.columns)}, inplace=True)

    value_d = dict()
    for l, (i, row) in zip(['A','B','C','D','E','F','G','H'], df.iterrows()):
        for i, value in enumerate(row):
            key = (retest_pla,replicate,f'{l}{str(i+1).zfill(2)}')
            res[key] = value

In [9]:
# Read in the plate layout file
apwt_df = pd.read_csv(f'{inputs_dir}/yeast_N2H/mapping_yn2h.csv')
apwt_df = apwt_df[apwt_df["standard_batch"] == 'yN2Hapwt']

col = ['ad_orf_id', 'ad_symbol', 'db_orf_id','db_symbol', 'db_mut_id', 'variant_id','retest_pla','retest_pos',
       'N2H_raw_r01', 'N2H_raw_r02']

apwt_df = apwt_df[col].copy()

In [10]:
# Get list of RC4 mut_ids
rc4_mut_ids = pd.read_csv(f"{allele_inputs}/VarChamp_Consolidated_allele_collection.tsv", sep="\t")
rc4_mut_ids = rc4_mut_ids[rc4_mut_ids.collection == 'RC4'].copy()
rc4_mut_ids = set(zip(rc4_mut_ids.orf_id, rc4_mut_ids.mutation_id_rc4))

In [11]:
# Read in file telling us which wells were empty and should be filtered out
empty_well_df = pd.read_excel(f'{inputs_dir}/yeast_N2H/20210328_yN2H_empty_wells.xlsx',sheet_name='Sheet2')
empty_well_df['replicate'] = empty_well_df.Plates.apply(lambda x: int(x.split('_')[2][-1]))
empty_well_df['plate'] = empty_well_df.Plates.apply(lambda x: int(x.split('_')[-1].replace('Plate','')))
empty_well_df['empty_wells'] = empty_well_df['Empty wells'].apply(lambda x: {f"{e.replace(' ','')[0]}{e.replace(' ','')[1:].zfill(2)}" for e in x.split(',')})
empty_well_set =  {(plate, rep, well) for plate, rep, wells in zip(empty_well_df.plate, empty_well_df.replicate, empty_well_df.empty_wells) for well in wells }

apwt_df['empty_well_rep1'] = [(db_orf_id, db_mut_id) in rc4_mut_ids or 
                              (plate,1,retest_pos) in empty_well_set 
                              for db_orf_id, db_mut_id, plate, retest_pos in zip(apwt_df.db_orf_id, apwt_df.db_mut_id, apwt_df.retest_pla, apwt_df.retest_pos)]

apwt_df['empty_well_rep2'] = [(db_orf_id, db_mut_id) in rc4_mut_ids or 
                              (plate,2,retest_pos) in empty_well_set 
                              for db_orf_id, db_mut_id, plate, retest_pos in zip(apwt_df.db_orf_id, apwt_df.db_mut_id, apwt_df.retest_pla, apwt_df.retest_pos)]


In [12]:
# Get scores for empty control wells
empty_N1_rep1_d = {(db,db_mut_id) if db_mut_id == db_mut_id else (db,999999):score for db, db_mut_id, score in zip(apwt_df[(apwt_df.variant_id == 'empty_N1')&(apwt_df.empty_well_rep1 == False)].db_orf_id, 
                                                                                                                    apwt_df[(apwt_df.variant_id == 'empty_N1')&(apwt_df.empty_well_rep1 == False)].db_mut_id, 
                                                                                                                    apwt_df[(apwt_df.variant_id == 'empty_N1')&(apwt_df.empty_well_rep1 == False)].N2H_raw_r01)}

empty_N2_rep1_d = dict(zip(apwt_df[(apwt_df.variant_id == 'empty_N2')&(apwt_df.empty_well_rep1 == False)].ad_orf_id, apwt_df[(apwt_df.variant_id == 'empty_N2')&(apwt_df.empty_well_rep1 == False)].N2H_raw_r01))

empty_N1_rep2_d = {(db,db_mut_id) if db_mut_id == db_mut_id else (db,999999):score for db, db_mut_id, score in zip(apwt_df[(apwt_df.variant_id == 'empty_N1')&(apwt_df.empty_well_rep2 == False)].db_orf_id, 
                                                                                                                    apwt_df[(apwt_df.variant_id == 'empty_N1')&(apwt_df.empty_well_rep2 == False)].db_mut_id, 
                                                                                                                    apwt_df[(apwt_df.variant_id == 'empty_N1')&(apwt_df.empty_well_rep2 == False)].N2H_raw_r02)}

empty_N2_rep2_d = dict(zip(apwt_df[(apwt_df.variant_id == 'empty_N2')&(apwt_df.empty_well_rep2 == False)].ad_orf_id, apwt_df[(apwt_df.variant_id == 'empty_N2')&(apwt_df.empty_well_rep2 == False)].N2H_raw_r02))

In [13]:
# Calculate normalized luminescence ratio
yN2H_normalized = apwt_df[apwt_df.variant_id.isin(['wt', 'allele', 'RRS','PRS'])].copy()

yN2H_normalized['db_orf_mut'] = [(orf_id,mut_id) if mut_id == mut_id else (orf_id,999999) for orf_id,mut_id in zip(yN2H_normalized.db_orf_id, yN2H_normalized.db_mut_id)]
yN2H_normalized['empty_N1_rep1'] = yN2H_normalized.db_orf_mut.map(empty_N1_rep1_d)
yN2H_normalized['empty_N2_rep1'] = yN2H_normalized.ad_orf_id.map(empty_N2_rep1_d)
yN2H_normalized['empty_N1_rep2'] = yN2H_normalized.db_orf_mut.map(empty_N1_rep2_d)
yN2H_normalized['empty_N2_rep2'] = yN2H_normalized.ad_orf_id.map(empty_N2_rep2_d)
yN2H_normalized.drop(columns='db_orf_mut',inplace=True)

yN2H_normalized['max_ctrl_r1'] = [max([en1,en2]) for en1, en2 in zip(yN2H_normalized.empty_N1_rep1, yN2H_normalized.empty_N2_rep1)]
yN2H_normalized['max_ctrl_r2'] = [max([en1,en2]) for en1, en2 in zip(yN2H_normalized.empty_N1_rep2, yN2H_normalized.empty_N2_rep2)]

yN2H_normalized['valid_r1'] = [ew == False and  en1 == en1 and en2 == en2 for ew, en1, en2 in zip(yN2H_normalized.empty_well_rep1, yN2H_normalized.empty_N1_rep1, yN2H_normalized.empty_N2_rep1)]
yN2H_normalized['valid_r2'] = [ew == False and  en1 == en1 and en2 == en2 for ew, en1, en2 in zip(yN2H_normalized.empty_well_rep2, yN2H_normalized.empty_N1_rep2, yN2H_normalized.empty_N2_rep2)]

yN2H_normalized['NLR1'] = [np.nan if not valid_r1 else N2H_raw_r01/max_ctrl_r1 for N2H_raw_r01,valid_r1,max_ctrl_r1 in zip(yN2H_normalized.N2H_raw_r01, yN2H_normalized.valid_r1, yN2H_normalized.max_ctrl_r1)]
yN2H_normalized['NLR2'] = [np.nan if not valid_r2 else N2H_raw_r02/max_ctrl_r2 for N2H_raw_r02,valid_r2,max_ctrl_r2 in zip(yN2H_normalized.N2H_raw_r02, yN2H_normalized.valid_r2, yN2H_normalized.max_ctrl_r2)]

mean_nlr_rrs_r1 = np.mean(yN2H_normalized[(yN2H_normalized.valid_r1)&(yN2H_normalized.variant_id == 'RRS')].NLR1)
mean_nlr_rrs_r2 = np.mean(yN2H_normalized[(yN2H_normalized.valid_r1)&(yN2H_normalized.variant_id == 'RRS')].NLR2)
std_nlr_rrs_r1 = np.std(yN2H_normalized[(yN2H_normalized.valid_r1)&(yN2H_normalized.variant_id == 'RRS')].NLR1)
std_nlr_rrs_r2 = np.std(yN2H_normalized[(yN2H_normalized.valid_r1)&(yN2H_normalized.variant_id == 'RRS')].NLR2)

d = {0:1,1:0}
yN2H_normalized['repeat_ratio'] = [[nlr1,nlr2][i%2]/[nlr1,nlr2][d[(i%2)]] for i,(nlr1, nlr2) in enumerate(zip(yN2H_normalized.NLR1, yN2H_normalized.NLR2))]
yN2H_normalized['repeat_ratio_log2'] = yN2H_normalized.repeat_ratio.apply(np.log2)

yN2H_normalized['avg_NLR'] = [np.mean([nlr1, nlr2]) for nlr1,nlr2 in zip(yN2H_normalized.NLR1, yN2H_normalized.NLR2)]

In [14]:
# Calculate perturbation z-scores
allele_df = yN2H_normalized.query("variant_id == 'allele' and ~avg_NLR.isnull()").copy()
df = yN2H_normalized.query("variant_id == 'wt'")

wt_NLR_d = {(r.db_orf_id, r.ad_orf_id):r.avg_NLR for i,r in df.iterrows()}
allele_df['avg_WT_NLR'] = [wt_NLR_d[N2,N1] for N2,N1 in zip(allele_df.db_orf_id, allele_df.ad_orf_id)]

allele_df = allele_df[~allele_df.avg_WT_NLR.isnull()].copy() #remove only one allele-WT
allele_df['AWT_ratio'] = allele_df.avg_NLR / allele_df.avg_WT_NLR

df = yN2H_normalized.query("variant_id == 'RRS'")
l = [np.mean([nlr1,nlr2]) for nlr1,nlr2 in zip(df.NLR1, df.NLR2) if nlr1 == nlr1 and nlr2 == nlr2]
std_rrs, mean_rrs = np.std(l), np.mean(l)
allele_df['z_WT_NLR'] = allele_df.avg_WT_NLR.apply(lambda x: (x-mean_rrs)/std_rrs)
allele_df['WT_detected'] = allele_df.z_WT_NLR.apply(lambda x: x >= nlr_detection_threshold)

std_ratio = np.std(yN2H_normalized.repeat_ratio_log2)
mean_ratio = np.mean(yN2H_normalized.repeat_ratio_log2)
allele_df['z_perturbation'] = allele_df.AWT_ratio.apply(lambda x: (np.log2(x)-mean_ratio)/std_ratio)
allele_df['z_cat'] = ['not_detected' if not wt_detected else 'increase_ppi' if zp>=2 else 'decrease_ppi' if zp<=-2 else 'no_perturbation' for zp,wt_detected in zip(allele_df.z_perturbation, allele_df.WT_detected)] 

In [20]:
# Incorporate sequence confirmation results
seq_N1 = pd.read_csv(f"{inputs_dir}/N2H_sequencing_confirmation/N2H_validation_nodes_validation_interactor.tsv", sep="\t")
seq_N2 = pd.read_csv(f"{inputs_dir}/N2H_sequencing_confirmation/N2H_validation_nodes_validation_allele.tsv", sep="\t")

seq_N1 = seq_N1[seq_N1["seq_confirmation"] == 1]
seq_N2 = seq_N2[seq_N2["seq_confirmation"] == 1]

seq_N1.rename(columns={"orf_id": "ad_orf_id"}, inplace=True)
seq_N2.rename(columns={"orf_id": "db_orf_id", "mut_id": "db_mut_id"}, inplace=True)

allele_df = allele_df.astype({"db_mut_id": int})

seq_N2

,db_orf_id,symbol,node_type,db_mut_id,N2H_GDEh_plate,N2H_GDN_plate,N2H_GDEh_well,seq_confirmation,nt_change,ccsb_mutation_id
0,12132,HCFC1,disease_wt,0,DisWTGDEh01,VUSN2HWT_GDN2h_01,B01,1,NaN,NaN
1,7515,XRCC4,disease_wt,0,DisWTGDEh01,VUSN2HWT_GDN2h_01,C01,1,NaN,NaN
2,71642,CACNB4,disease_wt,0,DisWTGDEh01,VUSN2HWT_GDN2h_01,E01,1,NaN,NaN
3,53479,POU4F3,disease_wt,0,DisWTGDEh01,VUSN2HWT_GDN2h_01,F01,1,NaN,NaN
4,53065,IL10,disease_wt,0,DisWTGDEh01,VUSN2HWT_GDN2h_01,H01,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
873,12573,DES,allele,203772,VUSN2Hmut_GDEh_10,VUSN2Hmut_GDN2h_10,E04,1,1009G>C,CCSBVarC006506
874,5613,APOA1,allele,215453,VUSN2Hmut_GDEh_10,VUSN2Hmut_GDN2h_10,G04,1,341T>C,CCSBVarC006764
875,3689,GCK,allele,214841,VUSN2Hmut_GDEh_10,VUSN2Hmut_GDN2h_10,B05,1,706G>A,CCSBVarC006690
876,100000400,PLA2G6,allele,221528,VUSN2Hmut_GDEh_10,VUSN2Hmut_GDN2h_10,C05,1,1442T>A,CCSBVarC007093


In [21]:
print(seq_N1['ad_orf_id'].dtype)
print(allele_df['ad_orf_id'].dtype)

print(seq_N2['db_orf_id'].dtype)
print(allele_df['db_orf_id'].dtype)

print(seq_N2['db_mut_id'].dtype)
print(allele_df['db_mut_id'].dtype)

int64
int64
int64
int64
int64
int64


In [ ]:
# Write out final file
allele_df.to_csv(f'{outputs_dir}/yN2H_apwt_allele_df.csv')